# FINC3014 — Performance Reconciliation Notebook

**You do not write code here.** You feed this notebook your data, run it, and read it.

**How to run (Google Colab, no install):**
1. Open <https://colab.research.google.com> then **File - Upload notebook** and pick this file.
2. Drag your `returns.csv` into Colab's **Files** pane (folder icon, left edge).
3. **Runtime - Run all.** Done.

**What it needs:** `returns.csv` in the same folder, three columns —
`date, portfolio_pct, benchmark_pct` — your window's daily returns **in per cent**,
copied out of your IBKR custom report's CSV export
(*ibkr-reports-guide.pdf, Section 5*). The file shipped with the template is
**example data**: replace it with yours (criterion B2).

**Your two jobs** are marked in the cells below: set `A` to your mandate's risk
aversion (**B3**), and fill in IBKR's printed numbers for the reconciliation
table (**B4**).

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv("returns.csv", parse_dates=["date"]).sort_values("date")
port = df["portfolio_pct"] / 100    # daily simple returns, as decimals
bench = df["benchmark_pct"] / 100

print(f"{len(df)} trading days: {df['date'].min().date()} -> {df['date'].max().date()}")
df.head()

## Step 1 — your parameters

`A` is **your client's risk aversion from your mandate** (brief, Section 2). Criterion
**B3** is marked on this being *your* value, not the template's.

`RF_ANNUAL` is the risk-free rate used in Sharpe/Sortino/CE. IBKR uses the historical
US 3-month T-bill; a fixed number here is fine — just remember it when your Sharpe
differs from theirs (that is a *reconciliation explanation*, not an error).

In [ ]:
A = 4.0            # <-- YOUR mandate's risk-aversion coefficient (2-10). B3.
RF_ANNUAL = 0.04   # annual risk-free rate used below (IBKR: 3-month T-bill)
TRADING_DAYS = 252 # trading days per year, for annualising

## Cumulative return

Compound the daily returns: $R_{cum} = \prod_t (1+r_t) - 1$.
This matches the last row of the report's **Cumulative** table.

In [ ]:
cum_port = (1 + port).prod() - 1
cum_bench = (1 + bench).prod() - 1
print(f"Cumulative return  portfolio: {cum_port:8.2%}   benchmark: {cum_bench:8.2%}")

## Volatility (standard deviation of returns)

Daily standard deviation, annualised: $\sigma_{ann} = \sigma_{daily}\sqrt{252}$.
**Reconciliation gotcha:** the IBKR report's *Standard Deviation* row is **daily**
— compare like with like.

In [ ]:
sd_daily_port, sd_daily_bench = port.std(), bench.std()
vol_ann_port = sd_daily_port * TRADING_DAYS ** 0.5
vol_ann_bench = sd_daily_bench * TRADING_DAYS ** 0.5
print(f"Daily std dev      portfolio: {sd_daily_port:8.2%}   benchmark: {sd_daily_bench:8.2%}")
print(f"Annualised vol     portfolio: {vol_ann_port:8.2%}   benchmark: {vol_ann_bench:8.2%}")

## Sharpe ratio

$\text{Sharpe} = \dfrac{\bar r_{ann} - r_f}{\sigma_{ann}}$ — excess return per unit
of total risk, with the annualised mean $\bar r_{ann} = \bar r_{daily}\times 252$.

In [ ]:
mean_ann_port = port.mean() * TRADING_DAYS
mean_ann_bench = bench.mean() * TRADING_DAYS
sharpe_port = (mean_ann_port - RF_ANNUAL) / vol_ann_port
sharpe_bench = (mean_ann_bench - RF_ANNUAL) / vol_ann_bench
print(f"Annualised mean    portfolio: {mean_ann_port:8.2%}   benchmark: {mean_ann_bench:8.2%}")
print(f"Sharpe ratio       portfolio: {sharpe_port:8.2f}   benchmark: {sharpe_bench:8.2f}")

## Sortino ratio

Same idea as Sharpe, but only **downside** deviation in the denominator — the client
is not upset by upside volatility:
$\text{Sortino} = \dfrac{\bar r_{ann} - r_f}{\sigma^{down}_{ann}}$,
where $\sigma^{down}$ is the standard deviation of the **negative** days only.

In [ ]:
def sortino(r, mean_ann):
    downside = r[r < 0]
    if len(downside) < 2:
        return float("nan")   # no losing days in the window
    dd_ann = downside.std() * TRADING_DAYS ** 0.5
    return (mean_ann - RF_ANNUAL) / dd_ann

sortino_port = sortino(port, mean_ann_port)
sortino_bench = sortino(bench, mean_ann_bench)
print(f"Sortino ratio      portfolio: {sortino_port:8.2f}   benchmark: {sortino_bench:8.2f}")

## Maximum drawdown

Grow one dollar through the window, track its running peak, and find the deepest
peak-to-trough fall: $\text{MDD} = \min_t\left(\frac{W_t}{\max_{s\le t} W_s} - 1\right)$.

In [ ]:
wealth_port = (1 + port).cumprod()
wealth_bench = (1 + bench).cumprod()
mdd_port = (wealth_port / wealth_port.cummax() - 1).min()
mdd_bench = (wealth_bench / wealth_bench.cummax() - 1).min()
print(f"Max drawdown       portfolio: {mdd_port:8.2%}   benchmark: {mdd_bench:8.2%}")

## The client's certainty-equivalent return (criterion B3)

$\mathrm{CE} = \bar r_{ann} - \tfrac{1}{2} A \sigma_{ann}^2$ — the guaranteed return
your client would trade your risky one for. **This is the yardstick the whole
assignment judges performance by** — a high-octane return that thrills a Hunter can
fail Dr Chen at the same P&L. Annualised terms throughout.

In [ ]:
ce_port = mean_ann_port - 0.5 * A * vol_ann_port ** 2
ce_bench = mean_ann_bench - 0.5 * A * vol_ann_bench ** 2
print(f"CE (A = {A:.0f})         portfolio: {ce_port:8.2%}   benchmark: {ce_bench:8.2%}")

## Cumulative return chart (criterion B5)

Growth of one dollar, portfolio vs benchmark. Right-click the figure to save it for
the README's results section.

In [ ]:
plt.figure(figsize=(9, 4.5))
plt.plot(df["date"], wealth_port, label="Portfolio")
plt.plot(df["date"], wealth_bench, label="Benchmark")
plt.xlabel("Date")
plt.ylabel("Growth of $1")
plt.title("Cumulative return over the trading window: portfolio vs benchmark")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## Step 2 — reconciliation against the IBKR report (criterion B4)

Open your **final** performance report (whole window). From the *Risk Measures
Benchmark Comparison* page and the *Cumulative* table's last row, type IBKR's
printed **portfolio-column** numbers into the dictionary below (plain numbers:
percent values as percents, ratios as ratios), then re-run the cell.

In [ ]:
ibkr_reported = {                       # <-- type IBKR's printed numbers here (B4)
    "Cumulative return (%)": None,      # Cumulative table, last row
    "Std deviation, daily (%)": None,   # Risk Measures: Standard Deviation
    "Sharpe ratio": None,               # Risk Measures: Sharpe Ratio
    "Sortino ratio": None,              # Risk Measures: Sortino Ratio
    "Max drawdown (%)": None,           # Risk Measures: Max Drawdown
}

computed = {
    "Cumulative return (%)": round(cum_port * 100, 2),
    "Std deviation, daily (%)": round(sd_daily_port * 100, 2),
    "Sharpe ratio": round(sharpe_port, 2),
    "Sortino ratio": round(sortino_port, 2),
    "Max drawdown (%)": round(mdd_port * 100, 2),
}

recon = pd.DataFrame({"This notebook": computed, "IBKR report": ibkr_reported})
recon["Gap"] = [None if i is None else round(c - i, 2)
                for c, i in zip(recon["This notebook"], recon["IBKR report"])]
recon

### Explain the gaps — one line each (this cell is yours to edit)

Legitimate reasons the two columns differ (name the one that applies):

- **Annualisation** — our Sharpe/Sortino use annualised mean and volatility; check what the report used.
- **Risk-free rate** — we used `RF_ANNUAL`; IBKR uses the historical 3-month T-bill series.
- **Daily vs annualised std** — the report's Standard Deviation row is daily.
- **Return method** — IBKR reports time-weighted returns (TWR); with no deposits or withdrawals mid-window, TWR and simple compounding agree.
- **Rounding** — the report prints two decimals.

*Your explanations:*

1. ...
2. ...

**Done.** The numbers above are the ones your README's results section quotes
(criterion B6 checks they match).